# Corrected forecast: Australian antidiabetic prescriptions

**Companion to the [historical experiment](<Forecasting of Antidiabetic Drug Prescriptions.ipynb>).** That notebook and its outputs remain untouched. This analysis uses the same documented 204-month CSV, a declared 168/36 chronological split, and three annual forecast origins. The newly calculated figures below come from [the standard-library runner](forecasting/corrected_forecast.py) and its [machine-readable output](forecasting/corrected_results.json). The runner was executed with Python 3.14.5; these code cells were also run sequentially as a smoke check. No Jupyter kernel or saved cell outputs were used.

The objective is to forecast the next 12 monthly observations, updating the model only after those 12 actuals become available. Values in the CSV are the series used here; the source calls them antidiabetic drug prescriptions in Australia. Do not infer patient counts or clinical outcomes from this aggregate series.

## 1. Data provenance and access

Source: Marco Peixeiro, [*Time Series Forecasting in Python* companion repository](https://github.com/marcopeix/TimeSeriesForecastingInPython), [exact CSV](https://github.com/marcopeix/TimeSeriesForecastingInPython/blob/master/data/AusAntidiabeticDrug.csv). The source repository displays an [Apache 2.0 license](https://github.com/marcopeix/TimeSeriesForecastingInPython/blob/master/LICENSE.txt). The 4,621-byte CSV was downloaded solely for this analysis; it is **not committed** here. SHA-256: `d6f32528dbeabdad405b578d351375aca9eb6f0eee7b632c9a49b909f80741c5`.

To reproduce, save that exact CSV as `forecasting/AusAntidiabeticDrug.csv`, then run `python forecasting/corrected_forecast.py` from the repository root. No package installation is required for the runner. It rejects a changed checksum, missing months, invalid values or unexpected schema. The 204 rows have columns `ds` (monthly date, July 1991 through June 2008) and `y` (positive numeric series).

In [ ]:
from forecasting.corrected_forecast import load_series, seasonal_first_difference, fit, forecast, evaluate, metrics

dates, y = load_series()
print(len(y), dates[0], dates[-1], min(y), max(y))

## 2. Correct seasonal differencing

Let $B y_t=y_{t-1}$. Ordinary first difference is $(1-B)y_t=y_t-y_{t-1}$. A monthly seasonal first difference is $(1-B^{12})y_t=y_t-y_{t-12}$. Applying both gives

$$z_t=(1-B)(1-B^{12})y_t=y_t-y_{t-1}-y_{t-12}+y_{t-13},\quad t\ge13.$$

For example, at month 13, $z_{13}=y_{13}-y_{12}-y_1+y_0$. The historical <code>np.diff(y_diff, n=12)</code> instead applies $(1-B)^{12}$ to an already first-differenced series: it is a thirteenth-order adjacent difference of the original data, **not** lag-12 seasonal differencing. Its ADF result therefore does not establish $D=1$. Stationarity is a modelling assumption here, not proven by a diagnostic.

In [ ]:
z = seasonal_first_difference(y[:168])  # diagnostics never inspect the holdout
print(z[:13], z[13], y[13]-y[12]-y[1]+y[0])

## 3. Honest model selection

The candidate set is deliberately small: $\mathrm{SARIMA}(p,1,0)(0,1,0)_{12}$ for $p\in\{0,1,2\}$, with an intercept in the differenced equation. On the transformed series, estimate $z_t=c+\sum_{j=1}^{p}\phi_j z_{t-j}+\epsilon_t$ by ordinary least squares. Conditional Gaussian AIC is $n\log(\mathrm{RSS}/n)+2k$, up to a common additive constant, where $k=p+2$ includes intercept and variance. Every candidate uses the **same 153 transformed training observations** ($t=15,\ldots,167$); otherwise conditional AIC rankings would mix fit quality with different sample sizes. Select the lowest AIC once on the first 168 values, breaking exact ties toward lower $p$. Refit that selected order at each origin. The final 36 values never choose the order.

This conditional AIC ranking applies only to the fixed transformation, Gaussian likelihood convention and common 153-row sample. Its values are **not numerically comparable** with statsmodels exact state-space AIC or the original saved table. The historical notebook's saved top row (1,0,2,3), AIC 26.619, conflicts with its narrative selection (2,3,1,3) and later AIC 276.234; convergence and notebook state are unresolved. We do not call its selected tuple optimal.

In [ ]:
initial_fits = [fit(y[:168], p) for p in (0, 1, 2)]
for m in initial_fits:
    print(m['p'], m['n'], round(m['aic_relative'], 3))
selected = min(initial_fits, key=lambda m: (m['aic_relative'], m['p']))['p']
print('selected p:', selected)

**Executed runner output:** conditional AIC* is -77.364 for (p=0), -128.299 for (p=1), and -144.118 for (p=2), each with (n=153). Thus the declared rule selects (p=2). The star denotes omission of the same Gaussian constant for every candidate.

## 4. Rolling-origin evaluation and baselines

The initial training window is July 1991 through June 2005 (168 observations). Forecast July 2005 through June 2006 from origin 168, then forecast the next two annual blocks from origins 180 and 192. At each origin, only earlier observations enter the fit or baselines. The ordinary naive forecast repeats the most recent observation for all 12 months. The seasonal naive forecast repeats the last observed 12-month cycle. The conditional AR model recursively forecasts transformed $z_t$ and inverts the two differences:

$$\hat y_t=\hat z_t+\hat y_{t-1}+\hat y_{t-12}-\hat y_{t-13}.$$

Previously forecast values are used when a needed lag is inside the future block. No observed holdout value enters that block before its next annual origin.

In [ ]:
result = evaluate(y)
print('selected order:', result['selected_p'])
for name, values in result['overall_metrics'].items():
    print(name, {k: round(v, 3) for k, v in values.items()})

## 5. Results freshly calculated from predictions

$\mathrm{MAE}=n^{-1}\sum|y_t-\hat y_t|$; $\mathrm{RMSE}=\sqrt{n^{-1}\sum(y_t-\hat y_t)^2}$; $\mathrm{MAPE}=100n^{-1}\sum|y_t-\hat y_t|/|y_t|$. All 36 actual values are positive, so MAPE is defined, though it remains sensitive to small denominators. Errors are calculated on the same 36 months for all three methods.

| Method | MAE | RMSE | MAPE |
| --- | ---: | ---: | ---: |
| Ordinary naive | 3.178 | 4.148 | 14.545% |
| Seasonal naive | 2.658 | 3.154 | 12.687% |
| Selected SARIMA(2,1,0)(0,1,0)[12] | **1.795** | **2.156** | **8.869%** |

| Forecast origin | Naive MAPE | Seasonal naive MAPE | Selected model MAPE |
| ---: | ---: | ---: | ---: |
| 168 | 13.398% | 7.963% | 5.232% |
| 180 | 17.538% | 15.416% | 10.150% |
| 192 | 12.700% | 14.681% | 11.224% |

The selected model has lower error than both baselines on this particular 36-month holdout. The historical notebook reported 7.899% MAPE for a **different** SARIMA specification; that figure is a saved historical output, not a fresh comparison with this model. Only three annual origins and one series are evaluated, so the margin does not establish expected performance on future or other drug series.

![Full series and rolling holdout forecasts](forecasting/rolling_forecasts.svg)

The top panel marks the first holdout origin in red. The lower panel overlays the actual 36 values and the three methods' 12-step forecasts. The [JSON artifact](forecasting/corrected_results.json) records every prediction and per-origin metric so the table and graph can be checked independently.

## 6. Limits and lessons

- This is a **restricted conditional least-squares SARIMA family**. It does not reproduce the original 625-fit statsmodels search or its exact likelihood, seasonal AR/MA terms, optimizer behaviour, or confidence intervals. Do not compare the two AIC scales.
- The selected transformed AR(2) coefficients are estimated from a short series. This experiment has no prospective validation, exogenous predictors, interval calibration, or formal stability guarantee. Errors may change if the monthly process shifts.
- AIC is an in-sample model-selection criterion; the rolling holdout is the evidence for forecasting error. The holdout was inspected only after the candidate set and selection rule were fixed.
- The model uses annual updates; a monthly-updated deployment needs a different evaluation protocol. A strong next step is a predeclared exact-likelihood SARIMA and forecast-interval study in a pinned statsmodels environment, with the same untouched 36-month protocol.

Run `python -m unittest discover -s forecasting -p 'test_*.py' -v` for focused checks of the seasonal operator, equal AIC samples, metrics and first-origin leakage. The script and tests ran successfully with Python 3.14.5. Notebook code cells were smoke-tested sequentially with Python, but not in a Jupyter kernel.